# Netflix Data Analysis Using Python
## Auspify Technologies – Data Analysis Internship Tasks

**Dataset:** `Dataset.csv` (provided with this project)  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Jupyter Notebook

### Project objective
Complete the six tasks listed in the Auspify Technologies PDF:
1. Netflix Data Cleaning & Preparation
2. Content Type Analysis Dashboard
3. Country-Wise Netflix Content Analysis
4. Trend Analysis by Release Year
5. Content Rating & Genre Analysis
6. Netflix Business Insights Report

The PDF states that interns need to complete **any 4 out of 6 tasks**. This notebook includes all six for extended practice.

### How to use
1. Keep `Dataset.csv` in the same folder as this notebook, or update `DATA_PATH` below.
2. Run the notebook from top to bottom.
3. Review each output and chart.
4. Check the `reports` folder for exported CSV files.

**Important:** Do not assume that missing or placeholder values are meaningful categories. This notebook checks actual missing values and common placeholder text separately. Country and genre counts can count one title more than once when a title has multiple countries or genres.

## 0. Import libraries and load the dataset

First, install the required packages if they are not already installed.

In [ ]:
# Run this cell only if packages are not installed:
# %pip install pandas numpy matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

DATA_PATH = Path("Dataset.csv")
REPORT_DIR = Path("reports")
REPORT_DIR.mkdir(exist_ok=True)

df_raw = pd.read_csv(DATA_PATH)

print("Dataset shape:", df_raw.shape)
print("\nColumn names:")
print(df_raw.columns.tolist())

display(df_raw.head())

### Understand the dataset

`shape` returns the number of rows and columns. `info()` shows data types and non-null counts. `describe()` summarizes numeric and categorical columns.

In [ ]:
df_raw.info()
display(df_raw.describe(include="all").T)

## Task 1 (Easy) – Netflix Data Cleaning & Preparation

**PDF workflow:** import dataset; identify and handle missing values; remove duplicates and formatting inconsistencies; standardize Country, Rating and Type; export the cleaned dataset.

We will preserve the original DataFrame in `df_raw` and create a separate working copy named `df`.

In [ ]:
df = df_raw.copy()

# 1. Standardize column names and trim whitespace in text columns
df.columns = df.columns.str.strip().str.lower()

for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].astype("string").str.strip()

# 2. Convert common placeholder values to proper missing values.
# Review this list before applying it if your business rules differ.
placeholder_values = ["", "Not Given", "not given", "N/A", "NA", "null", "None"]
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].replace(placeholder_values, pd.NA)

print("Missing values by column:")
display(df.isna().sum().to_frame("missing_count"))

print("Duplicate rows before removal:", df.duplicated().sum())
df = df.drop_duplicates().copy()
print("Duplicate rows after removal:", df.duplicated().sum())
print("Shape after duplicate removal:", df.shape)

### Standardize selected columns and data types

- `type`: consistent title case, e.g. Movie / TV Show.
- `country`: trim extra spaces around country names.
- `rating`: consistent uppercase formatting.
- `release_year`: numeric year.
- `date_added`: datetime value; invalid or blank dates become `NaT`.

In [ ]:
if "type" in df.columns:
    df["type"] = df["type"].str.strip().str.casefold().map({"movie": "Movie", "tv show": "TV Show"}).fillna(df["type"])

if "country" in df.columns:
    df["country"] = df["country"].str.replace(r"\s+", " ", regex=True).str.strip()

if "rating" in df.columns:
    df["rating"] = df["rating"].str.strip().str.upper()

if "release_year" in df.columns:
    df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce").astype("Int64")

if "date_added" in df.columns:
    df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")

print("Data types after standardization:")
print(df.dtypes)
print("\nMissing values after standardization:")
display(df.isna().sum().to_frame("missing_count"))
display(df.head())

### Export the cleaned dataset

We do not automatically fill missing director, country or rating values because the correct replacement is not known from this dataset. Keeping them missing avoids inventing information.

In [ ]:
cleaned_path = Path("Netflix_Cleaned.csv")
df.to_csv(cleaned_path, index=False)
print("Cleaned dataset saved to:", cleaned_path.resolve())

## Task 2 (Easy) – Content Type Analysis Dashboard

**PDF workflow:** load cleaned data; calculate Movies and TV Shows; create visualizations; compare proportions; summarize findings.

In [ ]:
content_counts = df["type"].value_counts(dropna=False)
display(content_counts.to_frame("total_titles"))

total_titles = df["type"].notna().sum()
content_percentages = (df["type"].value_counts(normalize=True) * 100).round(2)
display(content_percentages.to_frame("percentage"))

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="type", order=df["type"].value_counts().index)
plt.title("Netflix Content: Movies vs TV Shows")
plt.xlabel("Content Type")
plt.ylabel("Number of Titles")
plt.tight_layout()
plt.savefig(REPORT_DIR / "task2_content_type_bar.png", dpi=150)
plt.show()

In [ ]:
type_counts = df["type"].value_counts()
plt.figure(figsize=(7, 7))
plt.pie(type_counts.values, labels=type_counts.index, autopct="%1.1f%%", startangle=90)
plt.title("Netflix Content Type Proportion")
plt.tight_layout()
plt.savefig(REPORT_DIR / "task2_content_type_pie.png", dpi=150)
plt.show()

### Task 2 – Findings

Use the actual output above when writing your final report. Do not copy example values from tutorials; the code calculates the counts and percentages from your file.

In [ ]:
print("Content type findings")
for content_type, count in type_counts.items():
    pct = count / type_counts.sum() * 100
    print(f"{content_type}: {count:,} titles ({pct:.2f}%)")

## Task 3 (Medium) – Country-Wise Netflix Content Analysis

**PDF workflow:** extract and clean country information; calculate content count by country; identify top countries; create charts and rankings; generate business insights.

A title may list more than one country. We split the country field and use `explode()` so each title-country combination is counted separately.

In [ ]:
country_df = df.dropna(subset=["country"]).copy()
country_df["country"] = country_df["country"].str.split(",")
country_df = country_df.explode("country")
country_df["country"] = country_df["country"].astype("string").str.strip()
country_df = country_df[country_df["country"].notna() & country_df["country"].ne("")]

country_counts = (
    country_df["country"]
    .value_counts()
    .rename_axis("Country")
    .reset_index(name="Title_Count")
)

display(country_counts.head(10))

In [ ]:
top_countries = country_counts.head(10).sort_values("Title_Count")
plt.figure(figsize=(11, 6))
sns.barplot(data=top_countries, x="Title_Count", y="Country")
plt.title("Top 10 Countries by Netflix Title Count")
plt.xlabel("Title-country records")
plt.ylabel("Country")
plt.tight_layout()
plt.savefig(REPORT_DIR / "task3_top_countries.png", dpi=150)
plt.show()

In [ ]:
# Compare Movies and TV Shows for the top 10 countries
country_type = pd.crosstab(country_df["country"], country_df["type"])
top_country_names = country_counts.head(10)["Country"]
country_type_top10 = country_type.reindex(top_country_names).fillna(0)

country_type_top10.plot(kind="bar", figsize=(12, 6))
plt.title("Movies vs TV Shows Across Top 10 Countries")
plt.xlabel("Country")
plt.ylabel("Title-country records")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(REPORT_DIR / "task3_country_type_comparison.png", dpi=150)
plt.show()

### Task 3 – Interpretation note

Because a title can have multiple countries, the country counts are **title-country records**, not unique global title counts. Do not add the country counts and describe the sum as the number of unique Netflix titles.

In [ ]:
if not country_counts.empty:
    print("Top country in this dataset:", country_counts.iloc[0]["Country"])
    print("Title-country records:", int(country_counts.iloc[0]["Title_Count"]))
else:
    print("No country values are available after cleaning.")

## Task 4 (Medium) – Trend Analysis by Release Year

**PDF workflow:** organize content by release year; calculate yearly releases; identify growth and decline trends; create trend visualizations; interpret patterns.

`release_year` means the year the movie or TV show was originally released. It is not the date when Netflix added it.

In [ ]:
year_data = df.dropna(subset=["release_year"]).copy()
yearly_counts = (
    year_data.groupby("release_year")
    .size()
    .rename("Title_Count")
    .reset_index()
    .sort_values("release_year")
)
display(yearly_counts.tail(15))

In [ ]:
plt.figure(figsize=(13, 6))
sns.lineplot(data=yearly_counts, x="release_year", y="Title_Count", marker="o")
plt.title("Netflix Titles by Original Release Year")
plt.xlabel("Release Year")
plt.ylabel("Number of Titles")
plt.tight_layout()
plt.savefig(REPORT_DIR / "task4_release_year_trend.png", dpi=150)
plt.show()

In [ ]:
# Movies vs TV Shows by release year
year_type = (
    year_data.groupby(["release_year", "type"])
    .size()
    .reset_index(name="Title_Count")
)

plt.figure(figsize=(13, 6))
sns.lineplot(data=year_type, x="release_year", y="Title_Count", hue="type")
plt.title("Movies vs TV Shows by Release Year")
plt.xlabel("Release Year")
plt.ylabel("Number of Titles")
plt.tight_layout()
plt.savefig(REPORT_DIR / "task4_release_year_by_type.png", dpi=150)
plt.show()

In [ ]:
# Year-over-year change between consecutive years present in the data
yearly_counts["Previous_Year_Count"] = yearly_counts["Title_Count"].shift(1)
yearly_counts["Change_From_Previous_Row"] = (
    yearly_counts["Title_Count"] - yearly_counts["Previous_Year_Count"]
)
yearly_counts["Percentage_Change"] = (
    yearly_counts["Change_From_Previous_Row"]
    / yearly_counts["Previous_Year_Count"].replace(0, np.nan)
) * 100

display(yearly_counts.tail(15))

### Optional: Netflix additions by calendar year

This is a different question from original release-year trends. Use `date_added` to study when titles were added to Netflix.

In [ ]:
if "date_added" in df.columns:
    added = df.dropna(subset=["date_added"]).copy()
    added["added_year"] = added["date_added"].dt.year
    added_year_counts = (
        added.groupby("added_year")
        .size()
        .rename("Titles_Added")
        .reset_index()
        .sort_values("added_year")
    )
    display(added_year_counts)
    plt.figure(figsize=(11, 5))
    sns.lineplot(data=added_year_counts, x="added_year", y="Titles_Added", marker="o")
    plt.title("Titles Added to Netflix by Year")
    plt.xlabel("Year Added")
    plt.ylabel("Number of Titles")
    plt.tight_layout()
    plt.savefig(REPORT_DIR / "task4_netflix_added_year_trend.png", dpi=150)
    plt.show()

## Task 5 (Advanced) – Content Rating & Genre Analysis

**PDF workflow:** analyze rating categories; identify common genres; compare ratings across content types; create visualizations; present audience insights.

In [ ]:
rating_counts = df["rating"].value_counts().rename_axis("Rating").reset_index(name="Title_Count")
display(rating_counts)

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=rating_counts, x="Title_Count", y="Rating")
plt.title("Netflix Content Rating Distribution")
plt.xlabel("Number of Titles")
plt.ylabel("Rating")
plt.tight_layout()
plt.savefig(REPORT_DIR / "task5_rating_distribution.png", dpi=150)
plt.show()

In [ ]:
rating_type = pd.crosstab(df["rating"], df["type"])
display(rating_type)

rating_type.plot(kind="bar", figsize=(12, 6))
plt.title("Content Ratings by Type")
plt.xlabel("Rating")
plt.ylabel("Number of Titles")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(REPORT_DIR / "task5_rating_by_type.png", dpi=150)
plt.show()

In [ ]:
genre_df = df.dropna(subset=["listed_in"]).copy()
genre_df["listed_in"] = genre_df["listed_in"].str.split(",")
genre_df = genre_df.explode("listed_in")
genre_df["listed_in"] = genre_df["listed_in"].astype("string").str.strip()
genre_df = genre_df[genre_df["listed_in"].notna() & genre_df["listed_in"].ne("")]

genre_counts = (
    genre_df["listed_in"]
    .value_counts()
    .rename_axis("Genre")
    .reset_index(name="Title_Count")
)

display(genre_counts.head(15))

In [ ]:
top_genres = genre_counts.head(10).sort_values("Title_Count")
plt.figure(figsize=(11, 6))
sns.barplot(data=top_genres, x="Title_Count", y="Genre")
plt.title("Top 10 Netflix Genres")
plt.xlabel("Title-genre records")
plt.ylabel("Genre")
plt.tight_layout()
plt.savefig(REPORT_DIR / "task5_top_genres.png", dpi=150)
plt.show()

In [ ]:
genre_type = pd.crosstab(genre_df["listed_in"], genre_df["type"])
top_genre_names = genre_counts.head(10)["Genre"]
genre_type_top10 = genre_type.reindex(top_genre_names).fillna(0)

genre_type_top10.plot(kind="bar", figsize=(13, 6))
plt.title("Top Genres: Movies vs TV Shows")
plt.xlabel("Genre")
plt.ylabel("Title-genre records")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(REPORT_DIR / "task5_genre_by_type.png", dpi=150)
plt.show()

### Task 5 – Interpretation note

Ratings describe content classifications in the dataset. They do not directly measure popularity or viewer satisfaction. Genre counts are title-genre records because a title can have multiple genres.

## Task 6 (Advanced) – Netflix Business Insights Report

**PDF workflow:** perform complete EDA; identify content trends and patterns; develop multiple visual dashboards; generate actionable insights; prepare a final report.

The KPI values below are calculated from the cleaned dataset. Country and genre metrics are distinct counts or title-category records as labelled.

In [ ]:
total_titles = len(df)
movies = int((df["type"] == "Movie").sum())
tv_shows = int((df["type"] == "TV Show").sum())
movie_pct = (movies / total_titles * 100) if total_titles else 0
tv_pct = (tv_shows / total_titles * 100) if total_titles else 0

unique_countries = country_df["country"].nunique() if not country_df.empty else 0
unique_genres = genre_df["listed_in"].nunique() if not genre_df.empty else 0

kpi_summary = pd.DataFrame({
    "Metric": [
        "Total titles",
        "Movies",
        "TV Shows",
        "Movies percentage",
        "TV Shows percentage",
        "Distinct countries listed",
        "Distinct genres listed"
    ],
    "Value": [
        total_titles, movies, tv_shows,
        round(movie_pct, 2), round(tv_pct, 2),
        unique_countries, unique_genres
    ]
})
display(kpi_summary)

In [ ]:
# Simple KPI panel
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
kpi_values = [
    ("Total Titles", total_titles),
    ("Movies", movies),
    ("TV Shows", tv_shows)
]

for ax, (label, value) in zip(axes, kpi_values):
    ax.text(0.5, 0.55, f"{value:,}", ha="center", va="center", fontsize=25)
    ax.set_title(label, fontsize=13)
    ax.axis("off")

fig.suptitle("Netflix Dataset – KPI Overview", fontsize=16)
plt.tight_layout()
plt.savefig(REPORT_DIR / "task6_kpi_overview.png", dpi=150)
plt.show()

In [ ]:
# Build concise findings from the actual dataset
findings = [
    f"The dataset contains {total_titles:,} title records.",
    f"Movies account for {movie_pct:.2f}% of records and TV Shows account for {tv_pct:.2f}%.",
]

if not country_counts.empty:
    findings.append(
        f"The country with the highest title-country count is "
        f"{country_counts.iloc[0]['Country']} "
        f"({int(country_counts.iloc[0]['Title_Count']):,} title-country records)."
    )

if not genre_counts.empty:
    findings.append(
        f"The most frequently listed genre is {genre_counts.iloc[0]['Genre']} "
        f"({int(genre_counts.iloc[0]['Title_Count']):,} title-genre records)."
    )

if not rating_counts.empty:
    findings.append(
        f"The most common rating category is {rating_counts.iloc[0]['Rating']} "
        f"({int(rating_counts.iloc[0]['Title_Count']):,} titles)."
    )

for i, finding in enumerate(findings, start=1):
    print(f"{i}. {finding}")

In [ ]:
# Export KPI summary and business findings
kpi_summary.to_csv(REPORT_DIR / "Netflix_KPI_Summary.csv", index=False)

findings_df = pd.DataFrame({"Business_Finding": findings})
findings_df.to_csv(REPORT_DIR / "Netflix_Business_Insights.csv", index=False)

country_counts.to_csv(REPORT_DIR / "Country_Wise_Content_Count.csv", index=False)
genre_counts.to_csv(REPORT_DIR / "Genre_Wise_Content_Count.csv", index=False)
yearly_counts.to_csv(REPORT_DIR / "Release_Year_Trend.csv", index=False)

print("Reports and charts exported to:", REPORT_DIR.resolve())

## Final project conclusion – write after reviewing your outputs

Use your actual notebook outputs to write 4–6 evidence-based points:
- What share of records are Movies and TV Shows?
- Which countries have the highest title-country counts?
- Which release years show the highest content counts?
- Which ratings and genres appear most frequently?
- What data-quality limitations should a business user know?

Do not claim that this dataset proves audience preferences, viewing behaviour, revenue, or Netflix's full catalogue. The file contains title metadata, not viewership or financial data.

## Submission checklist (Auspify PDF)
- [ ] Project source code / notebook
- [ ] Screenshots of completed tasks
- [ ] GitHub repository link (if available)
- [ ] Project demonstration video (optional)